# Marine Sentinel — Persistent YOLO Training on Google Colab

This notebook keeps the dataset archive, checkpoints and final model in **your Google Drive**. Colab itself can be stopped safely; only its temporary runtime is lost. Select **Runtime → Change runtime type → T4 GPU** before training.

In [ ]:
!nvidia-smi
!pip -q install ultralytics
from google.colab import drive, files
from ultralytics import YOLO
import os, shutil, yaml

drive.mount('/content/drive')
DRIVE_PROJECT = '/content/drive/MyDrive/MarineSentinel'
DRIVE_DATASET = f'{DRIVE_PROJECT}/yolo_dataset.zip'
DRIVE_RUNS = f'{DRIVE_PROJECT}/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print(f'Persistent project folder: {DRIVE_PROJECT}')

## Save the dataset to Drive once

If the archive is not already in Drive, this cell asks you to upload it. Later Colab sessions reuse the saved Drive copy — no repeated upload.

In [ ]:
if not os.path.exists(DRIVE_DATASET):
    print('Upload dataset/yolo_dataset.zip from the VS Code project. This happens only once.')
    uploaded = files.upload()
    zip_name = next(name for name in uploaded if name.endswith('.zip'))
    shutil.move(zip_name, DRIVE_DATASET)
else:
    print('Using the dataset archive already saved in Google Drive.')
print(f'Dataset archive: {DRIVE_DATASET}')

In [ ]:
# Extract locally for fast GPU training. Re-run this after a new Colab session starts.
!rm -rf /content/marine_dataset
!unzip -q "$DRIVE_DATASET" -d /content/marine_dataset
dataset_root = '/content/marine_dataset/yolo'
data_config = {'path': dataset_root, 'train': 'images/train', 'val': 'images/valid',
               'test': 'images/test', 'names': {0: 'Crab-Pot', 1: 'Maybe-Crab-Pot'}}
with open('/content/dataset.yaml', 'w') as handle:
    yaml.safe_dump(data_config, handle, sort_keys=False)
print('Dataset extracted and configuration created.')

In [ ]:
# Fresh training. Checkpoints are written to Google Drive after every epoch.
# Use 30 epochs for today’s demo; change to 100 for the final model.
model = YOLO('yolo11n.pt')
model.train(data='/content/dataset.yaml', epochs=30, imgsz=640, batch=16, patience=15,
            optimizer='auto', cos_lr=True, degrees=8, translate=.08, scale=.30,
            fliplr=True, mosaic=.30, mixup=.05, seed=42, save_period=1,
            project=DRIVE_RUNS, name='marine_sentinel', exist_ok=True)

## Resume safely after Colab stops

Start a fresh Colab session, run the first four cells, then run this cell. Training continues from the last completed epoch instead of beginning again.

In [ ]:
last_checkpoint = f'{DRIVE_RUNS}/marine_sentinel/weights/last.pt'
assert os.path.exists(last_checkpoint), 'No saved checkpoint found; run Fresh training first.'
YOLO(last_checkpoint).train(resume=True)

In [ ]:
# Evaluate strictly on held-out test images, then download the permanent dashboard model.
best_path = f'{DRIVE_RUNS}/marine_sentinel/weights/best.pt'
best_model = YOLO(best_path)
metrics = best_model.val(data='/content/dataset.yaml', split='test', imgsz=640, conf=.25)
print(f'Test mAP50: {metrics.box.map50:.3f}')
print(f'Test mAP50-95: {metrics.box.map:.3f}')
files.download(best_path)